In [1]:
# do a), b)

$$Z = \frac{\bar{X} - \mu_0}{\frac{\sigma}{\sqrt{n}}}$$

$\bar{X}$ to średnia z próby,

$\mu_0$ to testowana wartość średnia,

$\sigma$ to znane odchylenie standardowe populacji,

$n$ to liczba obserwacji.

Wartość dystrybuanty dla tej statystyki obliczamy ze standardowego rozkładu normalnego $N(0,1)$.

Gdy odchylenie standardowe populacji nie jest znane (podpunkt b), używamy odchylenia standardowego wyliczonego z próby (oznaczanego jako $S$). Statystyka testowa to:

$$T = \frac{\bar{X} - \mu_0}{\frac{S}{\sqrt{n}}}$$

$S$ to odchylenie standardowe z próby (obliczane z nieobciążonego estymatora, czyli z dzielnikiem $n-1$).
Wartość dystrybuanty dla tej statystyki obliczamy z rozkładu t-Studenta z $n-1$ stopniami swobody.

In [2]:
# do c) d)

Szukamy największego $\mu_0$, dla którego wartość dystrybuanty jest większa lub równa $0.95$.
Dla rozkładu normalnego warunek $F_Z(z) \ge 0.95$ oznacza, że wartość statystyki $Z$ musi być większa od kwantyla rzędu 0.95 ze standardowego rozkładu normalnego (oznaczmy go jako $z_{0.95}$):

$$\frac{\bar{X} - \mu_0}{\frac{\sigma}{\sqrt{n}}} \ge z_{0.95}$$


$$\mu_0 \le \bar{X} - z_{0.95} \frac{\sigma}{\sqrt{n}}$$

Największe możliwe $\mu_0$ to zatem dokładnie $\bar{X} - z_{0.95} \frac{\sigma}{\sqrt{n}}$.

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import norm, t

def calculate_z_cdf(sample_mean: float, mu_0: float, sigma: float, n: int) -> float:
    z_stat: float = (sample_mean - mu_0) / (sigma / np.sqrt(n))
    return float(norm.cdf(z_stat))

def calculate_t_cdf(sample_mean: float, mu_0: float, s: float, n: int) -> float:
    t_stat: float = (sample_mean - mu_0) / (s / np.sqrt(n))
    return float(t.cdf(t_stat, df=n-1))

def find_max_mu0_z(sample_mean: float, sigma: float, n: int, prob: float) -> float:
    z_quantile: float = norm.ppf(prob)
    return float(np.round(sample_mean - z_quantile * (sigma / np.sqrt(n)), 2))

def find_max_mu0_t(sample_mean: float, s: float, n: int, prob: float) -> float:
    t_quantile: float = t.ppf(prob, df=n-1)
    return float(np.round(sample_mean - t_quantile * (s / np.sqrt(n)), 2))

df: pd.DataFrame = pd.read_csv('z1201.csv', header=None)
data: np.ndarray = df[0].to_numpy()

mu_0: float = 3.9
sigma: float = 1.0

n1: int = 10
n2: int = 20
n3: int = 40

data_n1: np.ndarray = data[:n1]
data_n2: np.ndarray = data[:n2]
data_n3: np.ndarray = data[:n3]

mean_n1: float = float(np.mean(data_n1))
mean_n2: float = float(np.mean(data_n2))
mean_n3: float = float(np.mean(data_n3))

s_n1: float = float(np.std(data_n1, ddof=1))
s_n2: float = float(np.std(data_n2, ddof=1))
s_n3: float = float(np.std(data_n3, ddof=1))

z_cdf_10: float = calculate_z_cdf(mean_n1, mu_0, sigma, n1)
z_cdf_20: float = calculate_z_cdf(mean_n2, mu_0, sigma, n2)
z_cdf_40: float = calculate_z_cdf(mean_n3, mu_0, sigma, n3)

t_cdf_10: float = calculate_t_cdf(mean_n1, mu_0, s_n1, n1)
t_cdf_20: float = calculate_t_cdf(mean_n2, mu_0, s_n2, n2)
t_cdf_40: float = calculate_t_cdf(mean_n3, mu_0, s_n3, n3)

max_mu0_z20: float = find_max_mu0_z(mean_n2, sigma, n2, 0.95)
max_mu0_t10: float = find_max_mu0_t(mean_n1, s_n1, n1, 0.95)

print(f"z_cdf_10 = {z_cdf_10}")
print(f"z_cdf_20 = {z_cdf_20}")
print(f"z_cdf_40 = {z_cdf_40}")
print(f"t_cdf_10 = {t_cdf_10}")
print(f"t_cdf_20 = {t_cdf_20}")
print(f"t_cdf_40 = {t_cdf_40}")
print(f"max_mu0_z20 = {max_mu0_z20}")
print(f"max_mu0_t10 = {max_mu0_t10}")

z_cdf_10 = 0.8655737493878928
z_cdf_20 = 0.9410569958642803
z_cdf_40 = 0.9864965464801648
t_cdf_10 = 0.8511847712210283
t_cdf_20 = 0.9376399904844838
t_cdf_40 = 0.9865953714048274
max_mu0_z20 = 3.88
max_mu0_t10 = 3.67


Testowanie hipotezy ze znamy std ($\sigma = 1$)

Dla 10 obserwacji: $F_Z(z_{10}) = 0.8656$

Dla 20 obserwacji: $F_Z(z_{20}) = 0.9411$

Dla 40 obserwacji: $F_Z(z_{40}) = 0.9865$

a)
Zgodnie z treścią zadania, odrzucamy hipotezę zerową $H_0: \mu = 3.9$, jeżeli wartość dystrybuanty jest większa lub równa $0.9$.

W przypadku 10 obserwacji wartość ta (ok. 0.866) jest mniejsza niż 0.9, więc nie ma podstaw do odrzucenia hipotezy $H_0$.

Przy 20 i 40 obserwacjach wartości (0.941 oraz 0.986) przekraczają próg 0.9, co oznacza, że odrzucamy hipotezę $H_0$ na rzecz hipotezy alternatywnej. Pokazuje to klasyczną właściwość testów statystycznych: im większa próba, tym test ma wyższą moc (większą zdolność do wykrycia, że prawdziwa średnia w populacji, z której losowano plik z1201.csv – oscylująca wokół 4.25 – różni się od testowanej wartości 3.9).

Testowanie hipotezy z nieznanym std

Dla 10 obserwacji: $F_T(t_{10}) = 0.8512$

Dla 20 obserwacji: $F_T(t_{20}) = 0.9376$

Dla 40 obserwacji: $F_T(t_{40}) = 0.9866$

Interpretacja (Podpunkt b):
Wnioski są tu niemal identyczne jak wyżej. Przy 10 obserwacjach nie odrzucamy $H_0$ (0.851 < 0.9), natomiast przy 20 i 40 obserwacjach odrzucamy ją (wartości > 0.9). Różnica polega na tym, że statystyka $T$ opiera się na odchyleniu standardowym z próby, a co za tym idzie, wartości rozkładu t-Studenta mają "grubsze ogony". Przy małej próbie (n=10) prowadzi to do nieco niższej wartości dystrybuanty w stosunku do rozkładu normalnego, ale w miarę wzrostu n (n=40) wartości te zaczynają się ze sobą zrównywać.

Szukanie największej wartości $\mu_0$

Dla 20 obserwacji i znanego $\sigma=1$: Największe $\mu_0 = 3.88$

Dla 10 obserwacji i nieznanego $\sigma$: Największe $\mu_0 = 3.67$

Interpretacja (Podpunkt c i d):W tych podpunktach odwracamy problem – szukamy granicznej wartości średniej hipotetycznej ($\mu_0$), przy której wartość dystrybuanty będzie mniejsza niż $0.95$, co na gruncie weryfikacji hipotez (i odrzucenia prawostronnego) odpowiada znalezieniu dolnej granicy przedziału ufności dla pewnego poziomu ufności.

Wynik 3.88 oznacza, że gdybyśmy mieli próbkę wielkości 20 elementów (ze znaną wariancją), to maksymalna testowana średnia, dla której wartość dystrybuanty statystyki $Z$ wyniesie co najmniej 0.95, to właśnie 3.88. Gdybyśmy testowali hipotezę $H_0: \mu = 3.89$, wartość dystrybuanty spadłaby poniżej 0.95.

Wynik 3.67 działa analogicznie dla 10 elementów (nieznana wariancja i statystyka $T$). Ze względu na małą próbę i dodatkową niepewność wynikającą z estymacji odchylenia standardowego, ten "margines bezpieczeństwa" jest zauważalnie większy (wartość brzegowa jest niższa).

-----------
zadanie 6

(a) nie znamy sredniej

$$\chi^2_{stat} = \frac{(n-1)S^2}{\sigma_0^2}$$

$n$ to liczebność próby,

$S^2$ to wariancja z próby (obliczana z dzielnikiem $n-1$),

$\sigma_0^2$ to testowana wariancja (tutaj $16$)

Ta statystyka ma rozkład $\chi^2$ z $n-1$ stopniami swobody.

(b) znamy srednia (2.8)

$$\chi^2_{stat} = \sum_{i=1}^{n} \frac{(X_i - \mu)^2}{\sigma_0^2}$$

$$\sum_{i=1}^{n} \left(\frac{X_i - \mu}{\sigma_0}\right)^2 = \frac{nS^2_{obc}}{\sigma_0^2} + \left(\frac{\bar{X} - \mu}{\sigma_0/\sqrt{n}}\right)^2$$

gdzie $S^2_{obc}$ to obciążony estymator wariancji (dzielony przez $n$). My jednak policzymy to wprost ze wzoru definiującego sumę.

Ta statystyka ma rozkład $\chi^2$ z pełnymi $n$ stopniami swobody (nie tracimy stopnia swobody na estymację średniej).

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import chi2

def calculate_chi2_unknown_mu_cdf(data_array: np.ndarray, sigma_sq_0: float) -> float:
    n: int = len(data_array)
    sample_var: float = float(np.var(data_array, ddof=1))
    chi2_stat: float = ((n - 1) * sample_var) / sigma_sq_0
    return float(chi2.cdf(chi2_stat, df=n-1))

def calculate_chi2_known_mu_cdf(data_array: np.ndarray, mu: float, sigma_sq_0: float) -> float:
    n: int = len(data_array)
    chi2_stat: float = float(np.sum(((data_array - mu) ** 2) / sigma_sq_0))
    return float(chi2.cdf(chi2_stat, df=n))

df_z1206: pd.DataFrame = pd.read_csv('z1206.csv', header=None)
data_6: np.ndarray = df_z1206[0].to_numpy()

sigma_sq_0: float = 16.0
known_mu: float = 2.8

cdf_unknown_mu: float = calculate_chi2_unknown_mu_cdf(data_6, sigma_sq_0)
cdf_known_mu: float = calculate_chi2_known_mu_cdf(data_6, known_mu, sigma_sq_0)

print(f"cdf_unknown_mu = {cdf_unknown_mu}")
print(f"cdf_known_mu = {cdf_known_mu}")

cdf_unknown_mu = 0.8158131576534667
cdf_known_mu = 0.7712788210810537


$F_{\chi^2_{(19)}}(\chi^2_{stat}) \approx 0.8158$. 


$F_{\chi^2_{(20)}}(\chi^2_{stat}) \approx 0.7713$

$$p\text{-value} = 2 \cdot \min(F(\chi^2_{stat}), 1 - F(\chi^2_{stat}))$$

Przypadek (a): $p = 2 \cdot \min(0.8158, 1 - 0.8158) = 2 \cdot 0.1842 = \textbf{0.3684}$

Przypadek (b): $p = 2 \cdot \min(0.7713, 1 - 0.7713) = 2 \cdot 0.2287 = \textbf{0.4574}$  

nie ma podstaw do odrzucneia hipotezy

----

zadanie 9

(a) Znamy wariancje $\sigma_1^2$ i $\sigma_2^2$ (i przyjmujemy, że są równe wariancjom z próby $S_1^2$ i $S_2^2$) 
Skoro "znamy" wariancje populacyjne, korzystamy ze standardowego rozkładu normalnego $N(0,1)$. Statystyka testowa ma postać:

$$Z = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{\frac{\sigma_1^2}{n_1} + \frac{\sigma_2^2}{n_2}}}$$

gdzie $\bar{X}_1, \bar{X}_2$ to średnie z prób, a $n_1, n_2$ to ich liczebności.

(b) Nie znamy wariancji, ale zakładamy, że są równe ($\sigma_1^2 = \sigma_2^2$) 
W tej sytuacji musimy wyestymować wspólną wariancję dla obu prób. Używamy połączonego estymatora wariancji (często oznaczanego jako $S_p^2$):


$$S_p^2 = \frac{(n_1-1)S_1^2 + (n_2-1)S_2^2}{n_1 + n_2 - 2}$$

Statystyka testowa ma rozkład t-Studenta z $n_1 + n_2 - 2$ stopniami swobody:

$$T = \frac{\bar{X}_1 - \bar{X}_2}{S_p \sqrt{\frac{1}{n_1} + \frac{1}{n_2}}}$$

Testowanie równości wariancji ($\sigma_1^2 = \sigma_2^2$)Do porównywania wariancji wykorzystujemy statystykę o rozkładzie F (Snedecora), która jest ilorazem dwóch wariancji.

(c) Zakładamy, że znamy średnie $\mu_1, \mu_2$ (i są one równe średnim z kolumn) Gdy średnia populacyjna jest znana, nie tracimy stopnia swobody na jej estymację. Wariancję liczymy "klasycznie" dzieląc przez $n$ (zamiast $n-1$):

$$\hat{\sigma}^2 = \frac{1}{n} \sum_{i=1}^n (X_i - \mu)^2$$

Statystyka testowa $F = \frac{\hat{\sigma}_1^2}{\hat{\sigma}_2^2}$ ma wtedy rozkład F z $n_1$ stopniami swobody w liczniku i $n_2$ w mianowniku. (Obliczane numerycznie wartości będą tu równe estymatorom obciążonym wariancji).

d) Nie znamy średnich $\mu_1, \mu_2$ 
To jest standardowy test. Ponieważ średnie są nieznane, estymujemy je z próby. Wariancję liczymy przy użyciu nieobciążonego estymatora (dzieląc przez $n-1$):

$$S^2 = \frac{1}{n-1} \sum_{i=1}^n (X_i - \bar{X})^2$$

Statystyka $F = \frac{S_1^2}{S_2^2}$ ma w tym wypadku rozkład F z $n_1-1$ oraz $n_2-1$ stopniami swobody.

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import norm, t, f

df: pd.DataFrame = pd.read_csv('z1209.csv', header=None)
data1: np.ndarray = df[0].to_numpy()
data2: np.ndarray = df[1].to_numpy()

n1: int = len(data1)
n2: int = len(data2)

mean1: float = float(np.mean(data1))
mean2: float = float(np.mean(data2))

var1_unbiased: float = float(np.var(data1, ddof=1))
var2_unbiased: float = float(np.var(data2, ddof=1))

var1_biased: float = float(np.var(data1, ddof=0))
var2_biased: float = float(np.var(data2, ddof=0))

z_stat: float = (mean1 - mean2) / np.sqrt(var1_unbiased / n1 + var2_unbiased / n2)
res_a: float = float(1.0 - norm.cdf(z_stat))

sp2: float = ((n1 - 1) * var1_unbiased + (n2 - 1) * var2_unbiased) / (n1 + n2 - 2)
t_stat: float = (mean1 - mean2) / np.sqrt(sp2 * (1 / n1 + 1 / n2))
res_b: float = float(1.0 - t.cdf(t_stat, df=n1 + n2 - 2))

f_stat_c: float = var1_biased / var2_biased
res_c: float = float(1.0 - f.cdf(f_stat_c, dfn=n1, dfd=n2))

f_stat_d: float = var1_unbiased / var2_unbiased
res_d: float = float(1.0 - f.cdf(f_stat_d, dfn=n1 - 1, dfd=n2 - 1))

print(f"res_a = {res_a}")
print(f"res_b = {res_b}")
print(f"res_c = {res_c}")
print(f"res_d = {res_d}")

res_a = 0.48176414378981025
res_b = 0.48182257802212114
res_c = 0.7443463591879098
res_d = 0.7416490485071047


Wynik (a) - test Z: 0.4818

Wynik (b) - test T: 0.4818

Skoro wartość $1 - F(x)$ wynosi u nas około 0.48, to znaczy, że nasza statystyka testowa wylądowała niezwykle blisko zera (jest tylko minimalnie dodatnia). Mówiąc prościej: średnie z obu kolumn pliku z1209.csv są niemal identyczne.

Wniosek: nie ma podstaw do odrzucneia hipotezy

Wynik (c) - znane średnie: 0.7443

Wynik (d) - nieznane średnie: 0.7416

Pamiętajmy, że nasz wynik to $1 - F(x) \approx 0.74$. Oznacza to, że sama dystrybuanta $F(x)$ to około 0.26. Skoro dystrybuanta jest mniejsza niż 0.5, to nasza obliczona statystyka $F$ wylądowała nieco po lewej stronie od szczytu krzywej (czyli jest nieco mniejsza od 1). Wskazuje to, że pierwsza próba ma odrobinę mniejszą wariancję niż druga, ale to odchylenie jest całkowicie naturalne i leży głęboko w głównym, bezpiecznym obszarze rozkładu (daleko od jakichkolwiek obszarów krytycznych, które zaczynałyby się np. przy wartościach < 0.05 lub > 0.95).

Wniosek: nie ma podstaw do odrzucenia hipotezy

----
zadanie 10

a)

$$Z_1 = \sqrt{-2 \ln(U_1)} \cos(2\pi U_2)$$

$$Z_2 = \sqrt{-2 \ln(U_1)} \sin(2\pi U_2)$$

Ponieważ w naszym pliku z1210.csv mamy 50 liczb, podzielimy je na dwie równe grupy po 25 ($U_1$ to pierwsza połowa, $U_2$ to druga połowa). Po zastosowaniu wzorów otrzymamy dwie paczki po 25 liczb z rozkładu normalnego, które na koniec połączymy z powrotem w wektor 50-elementowy.

b)
Rozkład wykładniczy $Exp(4)$
Tutaj korzystamy z potężnej i uniwersalnej metody odwracania dystrybuanty.
Dla rozkładu wykładniczego z parametrem $\lambda = 4$, gęstość to $f(x) = 4e^{-4x}$, a dystrybuanta wynosi:

$$F(x) = 1 - e^{-4x}$$

Aby wygenerować nową zmienną, przyrównujemy dystrybuantę do zmiennej losowej $U \sim U(0,1)$ i wyliczamy (odwracamy) $x$:

$$U = 1 - e^{-4x}$$

$$e^{-4x} = 1 - U$$

$$-4x = \ln(1 - U)$$

$$x = -\frac{1}{4}\ln(1 - U)$$

c) Rozkład o gęstości $f(x) = \frac{x}{8}$ na przedziale $[0, 4]$
Ponownie stosujemy metodę odwracania dystrybuanty. Najpierw musimy wyznaczyć dystrybuantę (czyli całkę z gęstości):

$$F(x) = \int_0^x \frac{t}{8} dt = \left[ \frac{t^2}{16} \right]_0^x = \frac{x^2}{16}$$

Teraz przyrównujemy ją do naszej zmiennej jednostajnej $U$ i wyliczamy $x$:

$$U = \frac{x^2}{16}$$

$$x = 4\sqrt{U}$$

In [6]:
import pandas as pd
import numpy as np

df_z1210: pd.DataFrame = pd.read_csv('z1210.csv', header=None)
u: np.ndarray = df_z1210[0].to_numpy()

half_n: int = len(u) // 2
u1: np.ndarray = u[:half_n]
u2: np.ndarray = u[half_n:]

z1: np.ndarray = np.sqrt(-2.0 * np.log(u1)) * np.cos(2.0 * np.pi * u2)
z2: np.ndarray = np.sqrt(-2.0 * np.log(u1)) * np.sin(2.0 * np.pi * u2)
norm_data: np.ndarray = np.concatenate((z1, z2))

norm_mean: float = float(np.mean(norm_data))
norm_var: float = float(np.var(norm_data, ddof=1))

exp_data: np.ndarray = -np.log(1.0 - u) / 4.0
exp_mean: float = float(np.mean(exp_data))
exp_var: float = float(np.var(exp_data, ddof=1))

custom_data: np.ndarray = 4.0 * np.sqrt(u)
custom_mean: float = float(np.mean(custom_data))
custom_var: float = float(np.var(custom_data, ddof=1))

print(f"norm_mean = {norm_mean}")
print(f"norm_var = {norm_var}")
print(f"exp_mean = {exp_mean}")
print(f"exp_var = {exp_var}")
print(f"custom_mean = {custom_mean}")
print(f"custom_var = {custom_var}")

norm_mean = -0.11930863792179512
norm_var = 1.45644226004845
exp_mean = 0.27000551085376745
exp_var = 0.09453952737734583
custom_mean = 2.6287426935879727
custom_var = 1.0130488259855395


1. Rozkład normalny $N(0,1)$ (Algorytm Boxa-Mullera)

Z definicji standardowego rozkładu normalnego wiemy, że:

Teoria: Wartość oczekiwana $\mu = 0$, Wariancja $\sigma^2 = 1$

U nas: Średnia $\approx -0.12$, Wariancja $\approx 1.46$

Interpretacja: Średnia wylądowała bardzo blisko zera, co jest świetnym sygnałem. Wariancja nieco "odskoczyła" (wynosi ok. 1.46 zamiast idealnego 1), ale nie jest to żaden błąd. Przy próbie liczącej zaledwie 50 wygenerowanych elementów, wariancja potrafi dość mocno falować. Jest to całkowicie naturalne zjawisko losowe.

2. Rozkład wykładniczy $Exp(4)$

Dla rozkładu wykładniczego z parametrem $\lambda = 4$ wzory na momenty to:

Teoria: Wartość oczekiwana to $\frac{1}{\lambda} = \frac{1}{4} = 0.25$. Wariancja to $\frac{1}{\lambda^2} = \frac{1}{16} = 0.0625$.

U nas: Średnia $\approx 0.27$, Wariancja $\approx 0.095$

Intepretacja: git

Rozkład z własną gęstością $f(x) = \frac{x}{8}$ na przedziale $[0, 4]$.

Tutaj musimy sobie na szybko policzyć wartości teoretyczne z całek:

Teoria:

Wartość oczekiwana: 

$E(X) = \int_0^4 x \cdot \frac{x}{8} dx = \int_0^4 \frac{x^2}{8} dx = \left[ \frac{x^3}{24} \right]_0^4 = \frac{64}{24} = \frac{8}{3} \approx 2.67$

Wariancja: 

Najpierw drugi moment $E(X^2) = \int_0^4 x^2 \cdot \frac{x}{8} dx = \left[ \frac{x^4}{32} \right]_0^4 = \frac{256}{32} = 8$. Zatem $V(X) = 8 - \left(\frac{8}{3}\right)^2 = 8 - \frac{64}{9} = \frac{8}{9} \approx 0.89$

Teoria: Średnia $\approx 2.67$, Wariancja $\approx 0.89$

U nas: Średnia $\approx 2.63$, Wariancja $\approx 1.01$